
# Análise de Evasão de Clientes — Chalenge Telecom X:

Este projeto tem como objetivo analisar os fatores que influenciam a evasão de clientes (churn) na empresa Telecom X. A empresa enfrenta um alto índice de cancelamentos e precisa entender os principais motivos que levam seus clientes a encerrar os serviços.

A análise foi conduzida a partir de dados obtidos via API, passando por um processo completo de ETL (Extração, Transformação e Carga), incluindo limpeza, tratamento e padronização dos dados para garantir sua qualidade e consistência.

Em seguida, foi realizada uma Análise Exploratória de Dados (EDA), com o uso de estatísticas descritivas e visualizações, buscando identificar padrões, tendências e relações entre as variáveis que possam explicar o comportamento de churn.

Por fim, o projeto apresenta insights relevantes e sugestões estratégicas que podem auxiliar a empresa na redução da evasão de clientes e na tomada de decisões orientadas por dados.


# 📌 Extração

In [ ]:

import requests
import pandas as pd

url = 'https://raw.githubusercontent.com/alura-cursos/challenge2-data-science/refs/heads/main/TelecomX_Data.json'

response = requests.get(url)
data = response.json()

df = pd.json_normalize(data)
df


In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe()

# 🔧 Transformação

## Padronização dos nomes das colunas

In [ ]:
# Padronizando os nomes das colunas:
df.columns = [col.lower().replace('.', '_') for col in df.columns]


In [ ]:
# Renomeando as colunas para português:
df.rename(columns={
    'customerid': 'cliente_id',
    'churn': 'evasao',
    'customer_gender': 'genero',
    'customer_seniorcitizen': 'idoso',
    'customer_partner': 'tem_parceiro',
    'customer_dependents': 'tem_dependentes',
    'customer_tenure': 'meses_contrato',
    'phone_phoneservice': 'servico_telefone',
    'phone_multiplelines': 'multiplas_linhas',
    'internet_internetservice': 'servico_internet',
    'internet_onlinesecurity': 'seguranca_online',
    'internet_onlinebackup': 'backup_online',
    'internet_deviceprotection': 'protecao_dispositivo',
    'internet_techsupport': 'suporte_tecnico',
    'internet_streamingtv': 'tv_streaming',
    'internet_streamingmovies': 'filmes_streaming',
    'account_contract': 'tipo_contrato',
    'account_paperlessbilling': 'fatura_digital',
    'account_paymentmethod': 'metodo_pagamento',
    'account_charges_monthly': 'cobranca_mensal',
    'account_charges_total': 'cobranca_total'
}, inplace=True)
df


## Verificações

In [ ]:
# Verificando valores únicos:
for col in df.columns:
  print(f'{col}: {df[col].unique()}')
  if df[col].nunique() < 50:
    print(df[col].unique())
    print('-' * 50)


In [ ]:
# Verificando se há valores nulos:
print("Numero de Nulos \n", df.isnull().sum())


In [ ]:
# Verificando se há valores duplicados:
print("Numero de duplicados", df.duplicated().sum())


In [ ]:
# Verificando se há espaços em branco:
df.apply(lambda x: x.astype(str).str.strip() == '').sum()


In [ ]:
# Verificando tipos de dados:
df.dtypes


## Transformações

In [ ]:
# Removendo espaços em branco:
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [ ]:
# Convertendo colunas de texto para categóricas:
colunas_categoricas = df.select_dtypes(include='object').columns

for col in colunas_categoricas:
    df[col] = df[col].astype('category')


In [ ]:
# Convertendo coluna 'idoso' (0/1 → nao/sim):
df['idoso'] = df['idoso'].map({0: 'nao', 1: 'sim'}).astype('category')


# Limpeza final e padronização:

In [ ]:
# Remover linhas com evasão vazia ou nula:
df = df[df['evasao'].notna()]
df['evasao'] = df['evasao'].astype(str).str.strip().str.lower()
df = df[df['evasao'] != '']
df = df[df['evasao'].isin(['yes', 'no'])]


In [ ]:
# Garantir tipo numérico em cobranca_total e remover nulos:
df['cobranca_total'] = pd.to_numeric(df['cobranca_total'], errors='coerce')
df = df.dropna(subset=['cobranca_total'])
print("Linhas após remoção de nulos em cobranca_total:", len(df))


In [ ]:
# Remover espaços e padronizar minúsculas em colunas de texto:
colunas_texto = df.select_dtypes(include=['object', 'category']).columns
for col in colunas_texto:
    df[col] = df[col].astype(str).str.strip().str.lower()


In [ ]:
# Tradução das colunas binárias (yes/no → sim/nao):
colunas_binarias = [
    'tem_parceiro', 'tem_dependentes', 'servico_telefone',
    'multiplas_linhas', 'seguranca_online', 'backup_online',
    'protecao_dispositivo', 'suporte_tecnico',
    'tv_streaming', 'filmes_streaming', 'fatura_digital'
]

for col in colunas_binarias:
    df[col] = df[col].replace({
        'yes': 'sim',
        'no': 'nao',
        'no phone service': 'sem servico de telefone',
        'no internet service': 'sem servico de internet'
    })


In [ ]:
# Tradução das colunas categóricas:
df['genero'] = df['genero'].replace({
    'female': 'feminino',
    'male': 'masculino'
})

df['tipo_contrato'] = df['tipo_contrato'].replace({
    'month-to-month': 'mensal',
    'one year': 'anual',
    'two year': 'bienal'
})

df['metodo_pagamento'] = df['metodo_pagamento'].replace({
    'electronic check': 'cheque_eletronico',
    'mailed check': 'cheque_correio',
    'bank transfer (automatic)': 'transferencia',
    'credit card (automatic)': 'cartao_credito'
})

df['servico_internet'] = df['servico_internet'].replace({
    'no': 'nao',
    'fiber optic': 'fibra',
    'dsl': 'dsl'
})

df['evasao'] = df['evasao'].replace({'yes': 'sim', 'no': 'nao'})


In [ ]:
# ✅ NOVO: Criando coluna de cobrança diária:
df['cobranca_diaria'] = df['cobranca_mensal'] / 30
df.head()


In [ ]:
# Check final:
print(df.isnull().sum())
print("\nDistribuição da evasão:")
print(df['evasao'].value_counts())


In [ ]:
df.head()

# 📊 Carregamento — Análise Exploratória do Churn:

## Análise Descritiva

In [ ]:
df.describe()

## Distribuição de Evasão

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
churn_counts = df['evasao'].value_counts()
churn_percentages = (churn_counts / len(df)) * 100

ax = churn_counts.plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Distribuição da Evasão de Clientes')
plt.xlabel('Evasão')
plt.ylabel('Quantidade de Clientes')
plt.xticks(rotation=0)

for i, (count, pct) in enumerate(zip(churn_counts, churn_percentages)):
    ax.text(i, count + 50, f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_distribuicao_evasao.png')
plt.show()


#### A taxa geral de evasão é de aproximadamente 26,5%, indicando que cerca de 1 em cada 4 clientes cancela o serviço.

## Evasão por Variáveis Categóricas

In [ ]:
import warnings
warnings.filterwarnings('ignore')

variaveis_categoricas = [
    'genero', 'idoso', 'tem_parceiro', 'tem_dependentes',
    'servico_telefone', 'servico_internet', 'tipo_contrato',
    'fatura_digital', 'metodo_pagamento'
]

fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for i, var in enumerate(variaveis_categoricas):
    sns.countplot(data=df, x=var, hue='evasao', ax=axes[i], palette=['steelblue', 'tomato'])
    axes[i].set_title(f'Evasão por {var}')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Quantidade')
    axes[i].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('grafico_variaveis_categoricas.png')
plt.show()


#### O grid acima permite comparar visualmente o comportamento de evasão em todas as variáveis categóricas. Destaque para tipo de contrato, tipo de internet e presença de dependentes como fatores diferenciadores.

## Evasão por Tempo de Contrato

In [ ]:
sns.histplot(data=df, x='meses_contrato', hue='evasao', bins=30, kde=True,
             palette=['steelblue', 'tomato'])

plt.title('Evasão por Tempo de Contrato')
plt.xlabel('Meses de Contrato')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.savefig('grafico_tempo_contrato.png')
plt.show()


#### Clientes com menor tempo de contrato apresentam maior taxa de evasão, indicando que o churn ocorre principalmente nos primeiros meses de relacionamento com a empresa.

In [ ]:
churn_tempo = df.groupby('meses_contrato')['evasao'].apply(
    lambda x: (x == 'sim').sum() / len(x) * 100
).reset_index()
churn_tempo.columns = ['meses_contrato', 'taxa_evasao']

plt.figure(figsize=(10, 5))
plt.plot(churn_tempo['meses_contrato'], churn_tempo['taxa_evasao'], color='tomato', marker='o', markersize=3)
plt.title('Taxa de Evasão (%) ao Longo do Tempo de Contrato')
plt.xlabel('Meses de Contrato')
plt.ylabel('Taxa de Evasão (%)')
plt.grid(True)
plt.tight_layout()
plt.savefig('grafico_taxa_evasao_tempo.png')
plt.show()


#### A taxa de evasão é significativamente maior nos primeiros meses de contrato, reduzindo conforme o tempo de permanência do cliente aumenta. Isso indica que o risco de churn está concentrado no início da jornada do cliente.

## Evasão por Tipo de Contrato

In [ ]:
sns.countplot(data=df, x='tipo_contrato', hue='evasao',
             palette=['steelblue', 'tomato'],
             order=['mensal', 'anual', 'bienal'])

plt.title('Evasão por Tipo de Contrato')
plt.xlabel('Tipo de Contrato')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.savefig('grafico_tipo_contrato.png')
plt.show()


#### Clientes com contrato mensal apresentam maior evasão em comparação com contratos anuais e bienais, sugerindo que a ausência de fidelização aumenta o risco de cancelamento.

## Evasão vs Cobrança Mensal

In [ ]:
sns.boxplot(data=df, x='evasao', y='cobranca_mensal',
           palette=['steelblue', 'tomato'])

plt.title('Evasão vs Cobrança Mensal')
plt.xlabel('Evasão')
plt.ylabel('Cobrança Mensal (R$)')
plt.tight_layout()
plt.savefig('grafico_cobranca_mensal.png')
plt.show()


#### Clientes que evadiram apresentam, em média, cobranças mensais mais elevadas, indicando que o preço pode ser um fator relevante na decisão de cancelamento.

## Evasão por Serviço de Internet

In [ ]:
sns.countplot(data=df, x='servico_internet', hue='evasao',
             palette=['steelblue', 'tomato'])

plt.title('Evasão por Tipo de Internet')
plt.xlabel('Serviço de Internet')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.savefig('grafico_servico_internet.png')
plt.show()


#### Clientes com internet fibra concentram maior taxa de evasão, o que pode indicar problemas de custo-benefício ou qualidade percebida do serviço.

## Tabelas de Percentual de Evasão

In [ ]:
# Taxa geral de evasão:
df['evasao'].value_counts(normalize=True) * 100


In [ ]:
# Evasão por tipo de contrato (%):
pd.crosstab(df['tipo_contrato'], df['evasao'], normalize='index') * 100


In [ ]:
# Evasão por gênero (%):
pd.crosstab(df['genero'], df['evasao'], normalize='index') * 100


In [ ]:
# Evasão por tipo de internet (%):
pd.crosstab(df['servico_internet'], df['evasao'], normalize='index') * 100


# 📋 Relatório Final

## 📊 INSIGHTS:

A análise exploratória identificou padrões claros associados à evasão de clientes. Clientes com menor tempo de contrato apresentam maior propensão ao churn, indicando que o cancelamento ocorre principalmente nos primeiros meses. A taxa geral de evasão de clientes é de aproximadamente 26,5%, enquanto 73,5% dos clientes permanecem na base, indicando que cerca de um quarto dos clientes cancela o serviço.

Ao analisar o gênero, observa-se que a evasão é bastante equilibrada entre homens (26,16%) e mulheres (26,92%), sugerindo que o gênero não é um fator determinante para o churn.

Além disso, clientes com contratos mensais possuem taxa de evasão significativamente maior em comparação com contratos anuais ou bienais, sugerindo baixa fidelização como fator relevante.

Observou-se também que clientes que cancelaram tendem a apresentar cobranças mensais mais elevadas, indicando sensibilidade ao preço.

Por fim, serviços específicos, como internet fibra, apresentam maior concentração de evasão, o que pode indicar problemas de qualidade percebida ou custo-benefício.


## 🧠 CONCLUSÃO:

A evasão de clientes na Telecom X está fortemente relacionada a três fatores principais: baixa fidelização (contratos mensais), tempo reduzido de relacionamento e maior custo mensal.

O padrão observado indica que clientes ainda não consolidados na base possuem maior risco de cancelamento, especialmente quando expostos a planos mais caros e sem incentivo de permanência.

A evasão de clientes na Telecom X apresenta uma taxa relevante, atingindo aproximadamente 1 em cada 4 clientes. No entanto, a análise indica que fatores demográficos simples, como gênero, não possuem impacto significativo no churn.

Isso sugere que a evasão está mais relacionada a fatores comportamentais e contratuais do que a características pessoais dos clientes.


## 💡 RECOMENDAÇÕES:

**1. Reduzir churn inicial:**

Criar ações de onboarding e acompanhamento nos primeiros meses, com foco em aumentar o engajamento e percepção de valor do cliente.

**2. Incentivar contratos mais longos:**

Oferecer benefícios para migração de planos mensais para anuais ou bienais, como descontos ou vantagens exclusivas.

**3. Revisar estratégia de preços:**

Avaliar planos com maior cobrança mensal, buscando ajustar o custo-benefício ou oferecer alternativas mais competitivas.

**4. Investigar serviços com maior evasão:**

Analisar qualitativamente serviços como internet fibra para identificar possíveis problemas de qualidade ou atendimento.

**5. Criar segmentação de risco:**

Desenvolver modelos futuros para identificar clientes com alta probabilidade de evasão e agir preventivamente.
